# 🚀 TMS Logistics — Algorithm Lab & Optimization Benchmark

Sổ tay này giúp bạn chạy và thử nghiệm toàn bộ hệ thống thuật toán tối ưu hóa vận tải (TMS):
- **VRP kết hợp Xếp dỡ thùng xe 2D (LIFO & Door Clearance)**
- **Benchmark 6 thuật toán metaheuristic** (Greedy, Hybrid ALNS, Packing-aware VNS, Tabu, ILS, Late Acceptance) trên 12 bộ dữ liệu thực tế & kịch bản phức tạp
- **Google OR-Tools Multi-start Fleet Routing Solver**
- **Báo cáo, so sánh chi phí & trực quan hóa đồ thị**

---

### 💡 Hướng dẫn tăng tốc trên Google Colab:
1. Vào menu **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Chọn **CPU High-RAM** hoặc **T4 GPU** (OR-Tools và các thuật toán tìm kiếm chạy song song đa tiến trình trên máy chủ Colab sẽ nhanh hơn máy cá nhân rất nhiều lần!).
3. Bấm **Save** (Lưu).


## 💻 1. Kiểm tra cấu hình phần cứng Google Colab
Chạy cell dưới đây để xem thông tin CPU, RAM và GPU của máy ảo:

In [ ]:
import os
import platform
import psutil

print('=' * 65)
print('💻 THÔNG TIN PHẦN CỨNG MÁY ẢO GOOGLE COLAB')
print('=' * 65)
print(f'Hệ điều hành: {platform.system()} {platform.release()}')
print(f'Python Version: {platform.python_version()}')
print(f'Số lượng vCPUs khả dụng: {os.cpu_count()}')
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'Dung lượng RAM: {ram_gb:.2f} GB')

try:
    import subprocess
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    print('\n🎮 GPU ĐƯỢC KÍCH HOẠT:')
    for line in gpu_info.split('\n')[7:11]:
        print(' ', line)
except Exception:
    print('\nℹ️ Chế độ CPU Runtime (rất tối ưu cho đa tiến trình Python / OR-Tools)')


## 📦 2. Cài đặt các thư viện phụ thuộc (Dependencies)
Cài đặt Google OR-Tools, FastAPI, Pydantic, NumPy, Pytest, Matplotlib, Seaborn...

In [ ]:
!pip install --quiet ortools>=9.12.0 numpy fastapi pydantic uvicorn pytest httpx tabulate matplotlib seaborn psutil

import ortools
print(f'✅ Google OR-Tools version: {ortools.__version__}')


## 📂 3. Nạp mã nguồn & Dữ liệu thuật toán
Chọn **Cách 1** (Khuyên dùng - Nhanh nhất) hoặc **Cách 2** (Git Clone) hoặc **Cách 3** (Google Drive):

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

WORKSPACE_DIR = '/content/tms_workspace'
os.makedirs(WORKSPACE_DIR, exist_ok=True)
%cd {WORKSPACE_DIR}

# Tự động phát hiện nếu file zip đã được kéo thả vào /content hoặc thư mục hiện tại
found_zip = None
for possible_zip in ['/content/tms_colab_bundle.zip', f'{WORKSPACE_DIR}/tms_colab_bundle.zip']:
    if os.path.exists(possible_zip):
        found_zip = possible_zip
        break

if found_zip:
    print(f'📦 Tìm thấy {found_zip}, tiến hành giải nén...')
    with zipfile.ZipFile(found_zip, 'r') as z:
        z.extractall(WORKSPACE_DIR)
    print('✅ Giải nén thành công!')
elif os.path.exists(f'{WORKSPACE_DIR}/algo_lab'):
    print('✅ Thư mục algo_lab đã tồn tại sẵn!')
else:
    print('ℹ️ Chưa có file zip. Hãy chạy Cell 3.1 bên dưới để tải file lên, hoặc Cell 3.2 để Git Clone.')


### 3.1. [Cách 1] Tải lên file `tms_colab_bundle.zip` (Chỉ ~400 KB)
*(Nếu bạn chưa tải file zip vào Colab, chạy cell dưới đây để chọn file từ máy)*

In [ ]:
from google.colab import files

if not os.path.exists(f'{WORKSPACE_DIR}/algo_lab'):
    print('Chọn file tms_colab_bundle.zip từ máy của bạn:')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall(WORKSPACE_DIR)
            print(f'🎉 Đã giải nén thành công {fname}!')
else:
    print('✅ Dữ liệu mã nguồn đã có đầy đủ, có thể chuyển sang bước tiếp theo.')


### 3.2. [Cách 2 - Tùy chọn] Clone từ Git Repository

In [ ]:
# Bỏ dấu # nếu muốn clone trực tiếp từ repository:
# !git clone -b feat/phase-1-core-tms https://github.com/Bang334/Project-Logictists.git /content/repo
# !cp -r /content/repo/algo_lab {WORKSPACE_DIR}/
# !cp -r /content/repo/optimizer {WORKSPACE_DIR}/


### 3.3. Cấu hình đường dẫn module (PYTHONPATH) & Kiểm tra kết nối

In [ ]:
import sys

for p in [WORKSPACE_DIR, os.path.join(WORKSPACE_DIR, 'optimizer'), os.path.join(WORKSPACE_DIR, 'algo_lab')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from algo_lab.common.models import OptimizationSolution
    from algo_lab.common.data_loader import load_dataset
    from optimizer.app.spatial_validator import SpatialValidator
    print('✅ Nạp thành công tất cả module: algo_lab, optimizer, spatial_validator!')
except Exception as exc:
    print('❌ Lỗi khi import module:', exc)
    print('Vui lòng kiểm tra lại xem file zip đã được giải nén đúng vào', WORKSPACE_DIR, 'chưa.')


## 🧪 4. Chạy kiểm tra nhanh hệ thống (Pytest Sanity Check)
Xác nhận các quy tắc nghiệp vụ TMS: Non-stackable packing, Door clearance, Time window, Costing...

In [ ]:
!pytest -q {WORKSPACE_DIR}/algo_lab/tests/


## ⚡ 5. Chạy So sánh nhanh (Fast Comparison)
So sánh tốc độ và chi phí giữa các thuật toán trên kịch bản Hà Nội (11 đơn hàng):

In [ ]:
!python {WORKSPACE_DIR}/algo_lab/run_fast_comparison.py


## 🎯 6. Chạy Benchmark trên 1 Kịch bản cụ thể
Bạn có thể thay đổi tên dataset bên dưới (ví dụ `hanoi_11_orders.json`, `db_danang_30_orders.json`, `scenario_1_multi_item.json`, `scenario_4_tight_windows.json`...)

In [ ]:
DATASET = 'algo_lab/datasets/hanoi_11_orders.json' # Thay đổi dataset tại đây
BUDGET = 2.0 # Ngân sách thời gian (giây)

!python {WORKSPACE_DIR}/algo_lab/run_benchmark.py --dataset {WORKSPACE_DIR}/{DATASET} --budget {BUDGET}


## 🔥 7. Chạy Benchmark Song Song Đa Tiến Trình (Parallel Benchmark)
**TẬN DỤNG TỐI ĐA SỨC MẠNH CỦA GOOGLE COLAB:**
- Chạy đối đầu **6 thuật toán**: `Greedy`, `Hybrid ALNS`, `Packing-aware VNS`, `Packing-aware Tabu`, `Packing-aware ILS`, `Packing-aware Late Acceptance`
- Trên **12 kịch bản**: 4 chi nhánh DB thật (Hà Nội, Đà Nẵng, Sóng Thần, Nha Trang) + 8 kịch bản thử thách gắt gao (nhiều kiện, hàng nặng, phân tán, time window chặt, đội xe hỗn hợp, tái sử dụng sàn, đa depot, phân mảnh cao).
- Số luồng (workers) tự động thích ứng theo số vCPU của Colab.

In [ ]:
import os

# Tối ưu hóa số luồng theo số core Colab
workers = max(2, os.cpu_count() or 4)
budget = 2.0 # Ngân sách thời gian tìm kiếm cho mỗi thuật toán/kịch bản (giây)

print(f'🚀 Bắt đầu Benchmark song song với {workers} workers, budget = {budget}s/lượt...')
!python {WORKSPACE_DIR}/algo_lab/run_parallel_benchmark.py --budget {budget} --workers {workers}


## 🏢 8. Chạy So sánh Toàn diện Hệ thống (Full System Comparison)
So sánh trực tiếp **Thuật toán hiện tại của dự án (Production Solver)** với các metaheuristics trên toàn bộ 4 chi nhánh thực tế:

In [ ]:
!python {WORKSPACE_DIR}/algo_lab/run_full_system_comparison.py


## 🚛 9. Đo lường Khả năng Mở rộng của Google OR-Tools (Scalability Benchmark)
Thử nghiệm bộ giải OR-Tools trên các bài toán quy mô lớn (50, 100, 200, 500 đơn hàng):

In [ ]:
!python {WORKSPACE_DIR}/optimizer/benchmark.py --sizes 50 100 200 --time-limit 5


## 📊 10. Trực quan hóa & Phân tích Đồ thị Kết quả
Đọc báo cáo đã xuất và vẽ biểu đồ so sánh chi phí và thời gian giữa các thuật toán:

In [ ]:
import glob
import os
import re
import matplotlib.pyplot as plt
import pandas as pd

# Tìm file báo cáo mới nhất
result_files = glob.glob(f'{WORKSPACE_DIR}/algo_lab/results/*.md')
if result_files:
    latest_file = max(result_files, key=os.path.getmtime)
    print(f'📈 Đọc dữ liệu từ file: {os.path.basename(latest_file)}')
    
    with open(latest_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    # In tóm tắt bảng kết quả
    table_lines = [l.strip() for l in lines if '|' in l]
    if table_lines:
        print('\n'.join(table_lines[:25]))
else:
    print('ℹ️ Hãy chạy Bước 7 hoặc Bước 8 để sinh file kết quả trước.')


## 💾 11. Tải toàn bộ kết quả về máy tính (Download Results)
Chạy cell dưới đây để nén thư mục `results/` và tự động tải về máy tính của bạn:

In [ ]:
import shutil
from google.colab import files

results_dir = os.path.join(WORKSPACE_DIR, 'algo_lab', 'results')
zip_path = '/content/tms_algo_results'

if os.path.exists(results_dir) and os.listdir(results_dir):
    shutil.make_archive(zip_path, 'zip', results_dir)
    print('🎉 Đang tải file tms_algo_results.zip về máy...')
    files.download(f'{zip_path}.zip')
else:
    print('⚠️ Thư mục results chưa có dữ liệu để tải về.')
